# Prepare Dual-Modality Rice Dataset for YOLOv11

This notebook scans `./datasets`, separates IR and normal images, creates a YOLO-compatible `train`/`val`/`test` layout, letterboxes every image to `640 x 640`, transforms labels to match the padded output, and applies strict geometric augmentation to the training split only.

The raw dataset is not modified. Prepared files are written to `./datasets_prepared/yolo_dual_640`.

In [ ]:
from __future__ import annotations

import hashlib
import math
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

try:
    import yaml
except ImportError as exc:
    raise ImportError('Install PyYAML first: pip install pyyaml') from exc

SEED = 42
SOURCE_ROOT = Path('datasets')
OUTPUT_ROOT = Path('datasets_prepared') / 'yolo_dual_640'
TARGET_SIZE = 640
SPLIT_RATIOS = {'train': 0.70, 'val': 0.20, 'test': 0.10}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
BACKGROUND_ONLY_HINTS = {'negative-background', 'negative_background', 'background-negative', 'background_only'}
AUGMENTATIONS = {
    'hflip': 'Horizontal flip',
    'vflip': 'Vertical flip',
    'rot90cw': 'Rotate 90 degrees clockwise',
    'rot90ccw': 'Rotate 90 degrees counter-clockwise',
    'rot180': 'Rotate 180 degrees',
}

random.seed(SEED)
np.random.seed(SEED)

print(f'Source root: {SOURCE_ROOT.resolve()}')
print(f'Output root: {OUTPUT_ROOT.resolve()}')

## Helpers

- Images are loaded with Pillow and `ImageOps.exif_transpose` so EXIF orientation is applied and then stripped by the rewritten output.
- Resizing uses black letterbox padding to avoid cropping or stretching.
- Labels can be YOLO boxes (`class x_center y_center width height`) or YOLO segmentation polygons (`class x1 y1 x2 y2 ...`). The current dataset uses segmentation polygons.

In [ ]:
@dataclass(frozen=True)
class Sample:
    image_path: Path
    label_path: Path | None
    modality: str
    stem: str
    background_only: bool = False


def detect_modality(path: Path) -> str | None:
    text = str(path).lower().replace('\\\\', '/')
    name = path.name.lower()
    parts = {part.lower() for part in path.parts}

    if any(token in parts for token in {'ir-image-datasets', 'ir_image_datasets', 'ir'}):
        return 'ir'
    if any(token in parts for token in {'normal-image-datasets', 'normal_image_datasets', 'normal'}):
        return 'normal'
    if '_ir' in name or '-ir' in name or name.startswith('ir_') or 'ir_jpg' in name:
        return 'ir'
    if 'normal' in name or 'white' in name or 'rgb' in name:
        return 'normal'
    return None


def find_label_for_image(image_path: Path) -> Path | None:
    candidates = [
        image_path.with_suffix('.txt'),
        image_path.parent.parent / 'labels' / f'{image_path.stem}.txt',
        image_path.parent / 'labels' / f'{image_path.stem}.txt',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def is_background_only_image(image_path: Path) -> bool:
    parts = {part.lower() for part in image_path.parts}
    return bool(parts.intersection(BACKGROUND_ONLY_HINTS))


def unique_output_stem(sample: Sample) -> str:
    digest = hashlib.sha1(str(sample.image_path.resolve()).encode('utf-8')).hexdigest()[:10]
    return f'{sample.stem}_{digest}'


def load_image_auto_oriented(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img).convert('RGB')
        return np.array(img)


def letterbox_rgb(image_rgb: np.ndarray, target_size: int = TARGET_SIZE) -> tuple[np.ndarray, dict[str, float]]:
    src_h, src_w = image_rgb.shape[:2]
    scale = min(target_size / src_w, target_size / src_h)
    new_w = int(round(src_w * scale))
    new_h = int(round(src_h * scale))
    resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    pad_x = (target_size - new_w) // 2
    pad_y = (target_size - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized

    meta = {
        'src_w': float(src_w),
        'src_h': float(src_h),
        'scale': float(scale),
        'pad_x': float(pad_x),
        'pad_y': float(pad_y),
        'target_size': float(target_size),
    }
    return canvas, meta


def save_rgb_image(path: Path, image_rgb: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    ok = cv2.imwrite(str(path), image_bgr)
    if not ok:
        raise IOError(f'Could not write image: {path}')

## Label Transform Helpers

The first transform maps source-normalized labels into the 640x640 letterboxed coordinate system. The augmentation transforms are exact square-coordinate operations, so labels remain synchronized with horizontal/vertical flips and 90/180 degree rotations.

In [ ]:
def clamp01(value: float) -> float:
    return min(1.0, max(0.0, value))


def format_float(value: float) -> str:
    return f'{clamp01(value):.8f}'.rstrip('0').rstrip('.')


def bbox_to_corners(xc: float, yc: float, w: float, h: float) -> list[tuple[float, float]]:
    x1, y1 = xc - w / 2.0, yc - h / 2.0
    x2, y2 = xc + w / 2.0, yc + h / 2.0
    return [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]


def corners_to_bbox(points: list[tuple[float, float]]) -> tuple[float, float, float, float]:
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    x1, x2 = clamp01(min(xs)), clamp01(max(xs))
    y1, y2 = clamp01(min(ys)), clamp01(max(ys))
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0, x2 - x1, y2 - y1)


def letterbox_point(x: float, y: float, meta: dict[str, float]) -> tuple[float, float]:
    target = meta['target_size']
    px = x * meta['src_w'] * meta['scale'] + meta['pad_x']
    py = y * meta['src_h'] * meta['scale'] + meta['pad_y']
    return clamp01(px / target), clamp01(py / target)


def transform_point_for_aug(x: float, y: float, aug_name: str | None) -> tuple[float, float]:
    if aug_name is None:
        return x, y
    if aug_name == 'hflip':
        return 1.0 - x, y
    if aug_name == 'vflip':
        return x, 1.0 - y
    if aug_name == 'rot90cw':
        return 1.0 - y, x
    if aug_name == 'rot90ccw':
        return y, 1.0 - x
    if aug_name == 'rot180':
        return 1.0 - x, 1.0 - y
    raise ValueError(f'Unknown augmentation: {aug_name}')


def transform_image_for_aug(image_rgb: np.ndarray, aug_name: str | None) -> np.ndarray:
    if aug_name is None:
        return image_rgb
    if aug_name == 'hflip':
        return cv2.flip(image_rgb, 1)
    if aug_name == 'vflip':
        return cv2.flip(image_rgb, 0)
    if aug_name == 'rot90cw':
        return cv2.rotate(image_rgb, cv2.ROTATE_90_CLOCKWISE)
    if aug_name == 'rot90ccw':
        return cv2.rotate(image_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if aug_name == 'rot180':
        return cv2.rotate(image_rgb, cv2.ROTATE_180)
    raise ValueError(f'Unknown augmentation: {aug_name}')


def transform_label_line(line: str, meta: dict[str, float], aug_name: str | None = None) -> str | None:
    parts = line.strip().split()
    if not parts:
        return None

    class_id = parts[0]
    values = [float(v) for v in parts[1:]]

    if len(values) == 4:
        corners = bbox_to_corners(*values)
        points = [letterbox_point(x, y, meta) for x, y in corners]
        points = [transform_point_for_aug(x, y, aug_name) for x, y in points]
        xc, yc, w, h = corners_to_bbox(points)
        if w <= 0 or h <= 0:
            return None
        return ' '.join([class_id, format_float(xc), format_float(yc), format_float(w), format_float(h)])

    if len(values) >= 6 and len(values) % 2 == 0:
        points = list(zip(values[0::2], values[1::2]))
        points = [letterbox_point(x, y, meta) for x, y in points]
        points = [transform_point_for_aug(x, y, aug_name) for x, y in points]
        flat = []
        for x, y in points:
            flat.extend([format_float(x), format_float(y)])
        return ' '.join([class_id, *flat])

    raise ValueError(f'Unsupported YOLO label row with {len(values)} numeric fields: {line[:120]}')


def transform_label_file(label_path: Path | None, output_path: Path, meta: dict[str, float], aug_name: str | None = None) -> int:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if label_path is None or not label_path.exists():
        output_path.write_text('', encoding='utf-8')
        return 0

    transformed_lines = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        transformed = transform_label_line(line, meta, aug_name=aug_name)
        if transformed is not None:
            transformed_lines.append(transformed)

    output_path.write_text('\n'.join(transformed_lines) + ('\n' if transformed_lines else ''), encoding='utf-8')
    return len(transformed_lines)

## Scan Source Dataset

Images are assigned to `ir` or `normal` by directory/name hints. The notebook looks for each image's `.txt` label next to the image or in a sibling `labels/` folder. Images placed under `datasets/negative-background/{normal,ir}/images` are treated as intentional background-only samples and receive empty label files.

In [ ]:
def scan_samples(source_root: Path) -> list[Sample]:
    samples = []
    for image_path in sorted(source_root.rglob('*')):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        modality = detect_modality(image_path)
        if modality is None:
            print(f'Skipping image with unknown modality: {image_path}')
            continue
        background_only = is_background_only_image(image_path)
        samples.append(
            Sample(
                image_path=image_path,
                label_path=None if background_only else find_label_for_image(image_path),
                modality=modality,
                stem=image_path.stem,
                background_only=background_only,
            )
        )
    return samples


samples = scan_samples(SOURCE_ROOT)
scan_summary = pd.DataFrame([
    {
        'modality': modality,
        'images': sum(s.modality == modality for s in samples),
        'with_labels': sum(s.modality == modality and s.label_path is not None for s in samples),
        'background_only': sum(s.modality == modality and s.background_only for s in samples),
        'unlabeled_non_background': sum(s.modality == modality and s.label_path is None and not s.background_only for s in samples),
    }
    for modality in ['normal', 'ir']
])
display(scan_summary)

unlabeled_non_background = [s for s in samples if s.label_path is None and not s.background_only]
if unlabeled_non_background:
    print('WARNING: Some non-background images have no label file and will be treated as empty labels:')
    for sample in unlabeled_non_background[:20]:
        print(f'- {sample.image_path}')
    if len(unlabeled_non_background) > 20:
        print(f'... and {len(unlabeled_non_background) - 20} more')

if not samples:
    raise RuntimeError(f'No images found under {SOURCE_ROOT.resolve()}')

## Split 70/20/10

The split is deterministic with `SEED = 42` and is performed independently for each modality so both IR and normal labeled subsets keep the same ratio as closely as possible. Intentional background-only samples are always assigned to `train` so they teach the detector to suppress false positives.

In [ ]:
def split_counts(total: int, ratios: dict[str, float]) -> dict[str, int]:
    train_count = int(math.floor(total * ratios['train']))
    val_count = int(math.floor(total * ratios['val']))
    test_count = total - train_count - val_count
    return {'train': train_count, 'val': val_count, 'test': test_count}


def split_by_modality(samples: list[Sample]) -> dict[str, dict[str, list[Sample]]]:
    rng = random.Random(SEED)
    result = {'normal': {'train': [], 'val': [], 'test': []}, 'ir': {'train': [], 'val': [], 'test': []}}

    for modality in result:
        modality_samples = [s for s in samples if s.modality == modality]
        background_samples = [s for s in modality_samples if s.background_only]
        labeled_samples = [s for s in modality_samples if not s.background_only]

        rng.shuffle(labeled_samples)
        rng.shuffle(background_samples)

        counts = split_counts(len(labeled_samples), SPLIT_RATIOS)
        train_end = counts['train']
        val_end = train_end + counts['val']
        result[modality]['train'] = labeled_samples[:train_end] + background_samples
        result[modality]['val'] = labeled_samples[train_end:val_end]
        result[modality]['test'] = labeled_samples[val_end:]
    return result


splits = split_by_modality(samples)
display(pd.DataFrame([
    {'modality': modality, 'split': split_name, 'images': len(split_samples)}
    for modality, split_map in splits.items()
    for split_name, split_samples in split_map.items()
]))

## Write Preprocessed and Augmented Dataset

This cell recreates `OUTPUT_ROOT`. Training samples, including intentional background-only samples, get the original letterboxed image plus five strict augmentations: horizontal flip, vertical flip, 90 CW, 90 CCW, and 180 rotation. Validation and test samples are only letterboxed.

In [ ]:
def prepare_output_dirs(output_root: Path) -> None:
    if output_root.exists():
        shutil.rmtree(output_root)
    for modality in ['normal', 'ir']:
        for split_name in ['train', 'val', 'test']:
            (output_root / modality / split_name / 'images').mkdir(parents=True, exist_ok=True)
            (output_root / modality / split_name / 'labels').mkdir(parents=True, exist_ok=True)


def write_one_variant(sample: Sample, split_name: str, image_rgb_640: np.ndarray, meta: dict[str, float], aug_name: str | None) -> dict[str, object]:
    suffix = 'orig' if aug_name is None else aug_name
    out_stem = f'{unique_output_stem(sample)}_{suffix}'
    images_dir = OUTPUT_ROOT / sample.modality / split_name / 'images'
    labels_dir = OUTPUT_ROOT / sample.modality / split_name / 'labels'
    image_out = images_dir / f'{out_stem}.jpg'
    label_out = labels_dir / f'{out_stem}.txt'

    transformed_image = transform_image_for_aug(image_rgb_640, aug_name)
    save_rgb_image(image_out, transformed_image)
    instances = transform_label_file(sample.label_path, label_out, meta, aug_name=aug_name)

    return {
        'modality': sample.modality,
        'split': split_name,
        'image': str(image_out),
        'label': str(label_out),
        'source_image': str(sample.image_path),
        'source_label': str(sample.label_path) if sample.label_path else None,
        'background_only': sample.background_only,
        'augmentation': suffix,
        'instances': instances,
    }


def write_dataset(splits: dict[str, dict[str, list[Sample]]]) -> pd.DataFrame:
    prepare_output_dirs(OUTPUT_ROOT)
    rows = []
    for modality, split_map in splits.items():
        for split_name, split_samples in split_map.items():
            for sample in split_samples:
                source_rgb = load_image_auto_oriented(sample.image_path)
                image_rgb_640, meta = letterbox_rgb(source_rgb, TARGET_SIZE)

                rows.append(write_one_variant(sample, split_name, image_rgb_640, meta, aug_name=None))

                if split_name == 'train':
                    for aug_name in AUGMENTATIONS:
                        rows.append(write_one_variant(sample, split_name, image_rgb_640, meta, aug_name=aug_name))
    return pd.DataFrame(rows)


manifest_df = write_dataset(splits)
manifest_path = OUTPUT_ROOT / 'manifest.csv'
manifest_df.to_csv(manifest_path, index=False)
print(f'Wrote manifest: {manifest_path.resolve()}')
display(manifest_df.head())

## Write YOLO Data YAML Files

One YAML file is created per modality. Adjust names here if your class order changes.

In [ ]:
CLASS_NAMES = {
    'normal': ['broken', 'chalky', 'damaged', 'discolored', 'foreign', 'paddy', 'red', 'whole'],
    'ir': ['broken', 'chalky', 'foreign', 'whole'],
}


for modality, names in CLASS_NAMES.items():
    yaml_payload = {
        'path': str((OUTPUT_ROOT / modality).resolve()),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(names),
        'names': names,
    }
    yaml_path = OUTPUT_ROOT / modality / 'data.yaml'
    yaml_path.write_text(yaml.safe_dump(yaml_payload, sort_keys=False), encoding='utf-8')
    print(f'Wrote {yaml_path}:')
    print(yaml_path.read_text(encoding='utf-8'))

## Final Summary

The summary reports generated image files and label instances per split. Training counts include the original plus all strict augmentations.

In [ ]:
summary = (
    manifest_df.groupby(['modality', 'split'], as_index=False)
    .agg(images=('image', 'count'), instances=('instances', 'sum'))
    .sort_values(['modality', 'split'])
)

source_summary = pd.DataFrame([
    {
        'modality': modality,
        'split': split_name,
        'source_images_before_augmentation': len(split_samples),
    }
    for modality, split_map in splits.items()
    for split_name, split_samples in split_map.items()
])

summary = summary.merge(source_summary, on=['modality', 'split'], how='left')
summary_path = OUTPUT_ROOT / 'summary.csv'
summary.to_csv(summary_path, index=False)

display(summary)
print(f'Summary CSV: {summary_path.resolve()}')
print(f'Prepared YOLO dataset root: {OUTPUT_ROOT.resolve()}')